# NCEP Reanalysis 2 数据下载脚本

自动下载1979-2014年的NCEP数据，支持多种变量

In [1]:
"""
NCEP Reanalysis 2 数据下载工具
支持下载uwnd, vwnd, omega, hgt等变量
时间范围：1979-2014
"""
import os
import requests
from pathlib import Path
import xarray as xr
from tqdm import tqdm
import time

In [2]:
def download_ncep_data(
    variable='uwnd',
    start_year=1979,
    end_year=2014,
    output_dir='./ncep_data',
    pressure_level=True,
    verify_download=True
):
    """
    下载NCEP Reanalysis 2 数据
    
    Parameters
    ----------
    variable : str
        变量名，如 'uwnd', 'vwnd', 'omega', 'hgt', 'air', 'rhum' 等
    start_year : int
        起始年份
    end_year : int
        结束年份
    output_dir : str
        输出目录
    pressure_level : bool
        是否为气压层数据（True）还是地表数据（False）
    verify_download : bool
        是否验证下载的文件可以正常打开
        
    Returns
    -------
    downloaded_files : list
        成功下载的文件列表
    """
    
    # 创建输出目录
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    # 设置基础URL
    if pressure_level:
        base_url = "https://downloads.psl.noaa.gov/Datasets/ncep.reanalysis2/Dailies/pressure/"
    else:
        base_url = "https://downloads.psl.noaa.gov/Datasets/ncep.reanalysis2/Dailies/surface/"
    
    downloaded_files = []
    failed_files = []
    
    print(f"{'='*70}")
    print(f"开始下载 {variable} 数据")
    print(f"时间范围: {start_year} - {end_year}")
    print(f"保存路径: {output_path.absolute()}")
    print(f"{'='*70}\n")
    
    # 循环下载每一年的数据
    for year in tqdm(range(start_year, end_year + 1), desc="下载进度"):
        # 构建文件名和URL
        filename = f"{variable}.{year}.nc"
        file_url = base_url + filename
        output_file = output_path / filename
        
        # 如果文件已存在且可以打开，跳过
        if output_file.exists():
            if verify_download:
                try:
                    ds = xr.open_dataset(output_file)
                    ds.close()
                    print(f"✓ {filename} 已存在且有效，跳过下载")
                    downloaded_files.append(str(output_file))
                    continue
                except Exception as e:
                    print(f"⚠ {filename} 已存在但损坏，重新下载...")
                    output_file.unlink()
            else:
                print(f"✓ {filename} 已存在，跳过下载")
                downloaded_files.append(str(output_file))
                continue
        
        # 下载文件
        try:
            print(f"⬇ 下载 {filename}...", end=" ")
            
            response = requests.get(file_url, stream=True, timeout=60)
            response.raise_for_status()
            
            # 获取文件大小
            total_size = int(response.headers.get('content-length', 0))
            
            # 写入文件
            with open(output_file, 'wb') as f:
                if total_size == 0:
                    f.write(response.content)
                else:
                    downloaded = 0
                    for chunk in response.iter_content(chunk_size=8192):
                        if chunk:
                            f.write(chunk)
                            downloaded += len(chunk)
            
            # 验证下载的文件
            if verify_download:
                try:
                    ds = xr.open_dataset(output_file)
                    ds.close()
                    print(f"✓ 成功 ({total_size / 1024 / 1024:.2f} MB)")
                    downloaded_files.append(str(output_file))
                except Exception as e:
                    print(f"✗ 文件损坏: {e}")
                    output_file.unlink()
                    failed_files.append(filename)
            else:
                print(f"✓ 成功 ({total_size / 1024 / 1024:.2f} MB)")
                downloaded_files.append(str(output_file))
            
            # 避免请求过快
            time.sleep(0.5)
            
        except requests.exceptions.RequestException as e:
            print(f"✗ 下载失败: {e}")
            failed_files.append(filename)
            if output_file.exists():
                output_file.unlink()
        
        except Exception as e:
            print(f"✗ 发生错误: {e}")
            failed_files.append(filename)
            if output_file.exists():
                output_file.unlink()
    
    # 打印汇总信息
    print(f"\n{'='*70}")
    print(f"下载完成！")
    print(f"成功: {len(downloaded_files)} 个文件")
    print(f"失败: {len(failed_files)} 个文件")
    if failed_files:
        print(f"失败的文件: {', '.join(failed_files)}")
    print(f"{'='*70}")
    
    return downloaded_files

## 使用示例

In [3]:
# 示例1: 下载850hPa纬向风（uwnd）
files_uwnd = download_ncep_data(
    variable='uwnd',
    start_year=1979,
    end_year=2024,
    output_dir='./ncep_data/uwnd',
    pressure_level=True,
    verify_download=True
)

开始下载 uwnd 数据
时间范围: 1979 - 2024
保存路径: /Users/lipu/Desktop/code/ncep_data/uwnd



下载进度:  35%|███▍      | 16/46 [00:01<00:02, 11.74it/s]

✓ uwnd.1979.nc 已存在且有效，跳过下载
✓ uwnd.1980.nc 已存在且有效，跳过下载
✓ uwnd.1981.nc 已存在且有效，跳过下载
✓ uwnd.1982.nc 已存在且有效，跳过下载
✓ uwnd.1983.nc 已存在且有效，跳过下载
✓ uwnd.1984.nc 已存在且有效，跳过下载
✓ uwnd.1985.nc 已存在且有效，跳过下载
✓ uwnd.1986.nc 已存在且有效，跳过下载
✓ uwnd.1987.nc 已存在且有效，跳过下载
✓ uwnd.1988.nc 已存在且有效，跳过下载
✓ uwnd.1989.nc 已存在且有效，跳过下载
✓ uwnd.1990.nc 已存在且有效，跳过下载
✓ uwnd.1991.nc 已存在且有效，跳过下载
✓ uwnd.1992.nc 已存在且有效，跳过下载
✓ uwnd.1993.nc 已存在且有效，跳过下载
✓ uwnd.1994.nc 已存在且有效，跳过下载
✓ uwnd.1995.nc 已存在且有效，跳过下载
✓ uwnd.1996.nc 已存在且有效，跳过下载
✓ uwnd.1997.nc 已存在且有效，跳过下载
✓ uwnd.1998.nc 已存在且有效，跳过下载
✓ uwnd.1999.nc 已存在且有效，跳过下载
✓ uwnd.2000.nc 已存在且有效，跳过下载
✓ uwnd.2001.nc 已存在且有效，跳过下载
✓ uwnd.2002.nc 已存在且有效，跳过下载
✓ uwnd.2003.nc 已存在且有效，跳过下载
✓ uwnd.2004.nc 已存在且有效，跳过下载
✓ uwnd.2005.nc 已存在且有效，跳过下载
✓ uwnd.2006.nc 已存在且有效，跳过下载
✓ uwnd.2007.nc 已存在且有效，跳过下载
✓ uwnd.2008.nc 已存在且有效，跳过下载
✓ uwnd.2009.nc 已存在且有效，跳过下载
✓ uwnd.2010.nc 已存在且有效，跳过下载
✓ uwnd.2011.nc 已存在且有效，跳过下载


下载进度:  74%|███████▍  | 34/46 [00:01<00:00, 27.92it/s]

✓ uwnd.2012.nc 已存在且有效，跳过下载
✓ uwnd.2013.nc 已存在且有效，跳过下载
✓ uwnd.2014.nc 已存在且有效，跳过下载
⬇ 下载 uwnd.2015.nc... ✓ 成功 (162.55 MB)
⬇ 下载 uwnd.2016.nc... 

...

⬇ 下载 uwnd.2023.nc... ✓ 成功 (151.46 MB)


下载进度:  98%|█████████▊| 45/46 [14:01<00:59, 59.44s/it]

⬇ 下载 uwnd.2024.nc... ✓ 成功 (152.31 MB)


下载进度: 100%|██████████| 46/46 [14:18<00:00, 18.66s/it]


下载完成！
成功: 46 个文件
失败: 0 个文件


In [4]:
# 示例2: 下载850hPa经向风（vwnd）
files_vwnd = download_ncep_data(
    variable='vwnd',
    start_year=1979,
    end_year=2024,
    output_dir='./ncep_data/vwnd',
    pressure_level=True,
    verify_download=True
)

开始下载 vwnd 数据
时间范围: 1979 - 2024
保存路径: /Users/lipu/Desktop/code/ncep_data/vwnd



下载进度:  35%|███▍      | 16/46 [00:00<00:00, 156.37it/s]

✓ vwnd.1979.nc 已存在且有效，跳过下载
✓ vwnd.1980.nc 已存在且有效，跳过下载
✓ vwnd.1981.nc 已存在且有效，跳过下载
✓ vwnd.1982.nc 已存在且有效，跳过下载
✓ vwnd.1983.nc 已存在且有效，跳过下载
✓ vwnd.1984.nc 已存在且有效，跳过下载
✓ vwnd.1985.nc 已存在且有效，跳过下载
✓ vwnd.1986.nc 已存在且有效，跳过下载
✓ vwnd.1987.nc 已存在且有效，跳过下载
✓ vwnd.1988.nc 已存在且有效，跳过下载
✓ vwnd.1989.nc 已存在且有效，跳过下载
✓ vwnd.1990.nc 已存在且有效，跳过下载
✓ vwnd.1991.nc 已存在且有效，跳过下载
✓ vwnd.1992.nc 已存在且有效，跳过下载
✓ vwnd.1993.nc 已存在且有效，跳过下载
✓ vwnd.1994.nc 已存在且有效，跳过下载
✓ vwnd.1995.nc 已存在且有效，跳过下载
✓ vwnd.1996.nc 已存在且有效，跳过下载
✓ vwnd.1997.nc 已存在且有效，跳过下载
✓ vwnd.1998.nc 已存在且有效，跳过下载
✓ vwnd.1999.nc 已存在且有效，跳过下载
✓ vwnd.2000.nc 已存在且有效，跳过下载
✓ vwnd.2001.nc 已存在且有效，跳过下载
✓ vwnd.2002.nc 已存在且有效，跳过下载
✓ vwnd.2003.nc 已存在且有效，跳过下载
✓ vwnd.2004.nc 已存在且有效，跳过下载
✓ vwnd.2005.nc 已存在且有效，跳过下载
✓ vwnd.2006.nc 已存在且有效，跳过下载
✓ vwnd.2007.nc 已存在且有效，跳过下载
✓ vwnd.2008.nc 已存在且有效，跳过下载
✓ vwnd.2009.nc 已存在且有效，跳过下载
✓ vwnd.2010.nc 已存在且有效，跳过下载
✓ vwnd.2011.nc 已存在且有效，跳过下载
✓ vwnd.2012.nc 已存在且有效，跳过下载


下载进度:  76%|███████▌  | 35/46 [00:00<00:00, 174.55it/s]

✓ vwnd.2013.nc 已存在且有效，跳过下载
✓ vwnd.2014.nc 已存在且有效，跳过下载
⬇ 下载 vwnd.2015.nc... 

...

⬇ 下载 vwnd.2023.nc... ✓ 成功 (161.04 MB)


下载进度:  98%|█████████▊| 45/46 [11:38<00:35, 35.36s/it]

⬇ 下载 vwnd.2024.nc... ✓ 成功 (161.63 MB)


下载进度: 100%|██████████| 46/46 [12:18<00:00, 16.05s/it]


下载完成！
成功: 46 个文件
失败: 0 个文件


In [5]:
# 示例3: 下载垂直速度（omega）
files_omega = download_ncep_data(
    variable='omega',
    start_year=1979,
    end_year=2024,
    output_dir='./ncep_data/omega',
    pressure_level=True,
    verify_download=True
)

开始下载 omega 数据
时间范围: 1979 - 2024
保存路径: /Users/lipu/Desktop/code/ncep_data/omega



下载进度:  37%|███▋      | 17/46 [00:00<00:00, 165.40it/s]

✓ omega.1979.nc 已存在且有效，跳过下载
✓ omega.1980.nc 已存在且有效，跳过下载
✓ omega.1981.nc 已存在且有效，跳过下载
✓ omega.1982.nc 已存在且有效，跳过下载
✓ omega.1983.nc 已存在且有效，跳过下载
✓ omega.1984.nc 已存在且有效，跳过下载
✓ omega.1985.nc 已存在且有效，跳过下载
✓ omega.1986.nc 已存在且有效，跳过下载
✓ omega.1987.nc 已存在且有效，跳过下载
✓ omega.1988.nc 已存在且有效，跳过下载
✓ omega.1989.nc 已存在且有效，跳过下载
✓ omega.1990.nc 已存在且有效，跳过下载
✓ omega.1991.nc 已存在且有效，跳过下载
✓ omega.1992.nc 已存在且有效，跳过下载
✓ omega.1993.nc 已存在且有效，跳过下载
✓ omega.1994.nc 已存在且有效，跳过下载
✓ omega.1995.nc 已存在且有效，跳过下载
✓ omega.1996.nc 已存在且有效，跳过下载
✓ omega.1997.nc 已存在且有效，跳过下载
✓ omega.1998.nc 已存在且有效，跳过下载
✓ omega.1999.nc 已存在且有效，跳过下载
✓ omega.2000.nc 已存在且有效，跳过下载
✓ omega.2001.nc 已存在且有效，跳过下载
✓ omega.2002.nc 已存在且有效，跳过下载
✓ omega.2003.nc 已存在且有效，跳过下载
✓ omega.2004.nc 已存在且有效，跳过下载
✓ omega.2005.nc 已存在且有效，跳过下载
✓ omega.2006.nc 已存在且有效，跳过下载
✓ omega.2007.nc 已存在且有效，跳过下载
✓ omega.2008.nc 已存在且有效，跳过下载
✓ omega.2009.nc 已存在且有效，跳过下载
✓ omega.2010.nc 已存在且有效，跳过下载
✓ omega.2011.nc 已存在且有效，跳过下载
✓ omega.2012.nc 已存在且有效，跳过下载
✓ omega.2013.nc 已存在且有效，跳过下载
✓ omega.2014.nc 已存在且

下载进度:  37%|███▋      | 17/46 [00:13<00:00, 165.40it/s]

✓ 成功 (109.88 MB)


...

⬇ 下载 omega.2023.nc... ✓ 成功 (179.40 MB)


下载进度:  98%|█████████▊| 45/46 [11:07<00:46, 46.03s/it]

⬇ 下载 omega.2024.nc... ✓ 成功 (180.03 MB)


下载进度: 100%|██████████| 46/46 [11:31<00:00, 15.02s/it]


下载完成！
成功: 46 个文件
失败: 0 个文件


## 验证下载的数据

In [6]:
# 验证并查看下载的数据
import xarray as xr

# 打开一个文件查看结构
if len(files_uwnd) > 0:
    ds = xr.open_dataset(files_uwnd[0])
    print("数据维度:", ds.dims)
    print("\n变量列表:")
    for var in ds.data_vars:
        print(f"  - {var}: {ds[var].dims}")
    print("\n坐标:")
    for coord in ds.coords:
        print(f"  - {coord}: {ds[coord].shape}")
    
    # 显示数据
    ds

数据维度: FrozenMappingWarningOnValuesAccess({'level': 17, 'lat': 73, 'lon': 144, 'time': 365, 'nbnds': 2})

变量列表:
  - time_bnds: ('time', 'nbnds')
  - uwnd: ('time', 'level', 'lat', 'lon')

坐标:
  - level: (17,)
  - lat: (73,)
  - lon: (144,)
  - time: (365,)


In [7]:
# 合并多年数据（可选）
def merge_yearly_files(file_list, output_file=None):
    """
    合并多个年份的nc文件为一个文件
    
    Parameters
    ----------
    file_list : list
        文件路径列表
    output_file : str, optional
        输出文件名，如果为None则不保存
        
    Returns
    -------
    ds : xarray.Dataset
        合并后的数据集
    """
    print(f"正在合并 {len(file_list)} 个文件...")
    
    # 打开并合并所有文件
    ds = xr.open_mfdataset(
        file_list,
        combine='by_coords',
        parallel=True,
        chunks={'time': 365}
    )
    
    print(f"合并完成！")
    print(f"时间范围: {ds.time.min().values} - {ds.time.max().values}")
    print(f"数据维度: {ds.dims}")
    
    # 保存合并后的文件
    if output_file:
        print(f"保存到 {output_file}...")
        ds.to_netcdf(output_file)
        print("保存完成！")
    
    return ds

# 示例：合并所有uwnd文件
# ds_uwnd_merged = merge_yearly_files(
#     files_uwnd, 
#     output_file='./ncep_data/uwnd_1979-2014.nc'
# )

# ds_vwnd_merged = merge_yearly_files(
#     files_vwnd, 
#     output_file='./ncep_data/vwnd_1979-2014.nc'
# )

## 常用NCEP变量列表

### 气压层数据 (pressure level)
- `uwnd` - 纬向风 (Zonal wind, m/s)
- `vwnd` - 经向风 (Meridional wind, m/s)
- `omega` - 垂直速度 (Vertical velocity, Pa/s)
- `air` - 气温 (Air temperature, K)
- `hgt` - 位势高度 (Geopotential height, m)
- `rhum` - 相对湿度 (Relative humidity, %)
- `shum` - 比湿 (Specific humidity, kg/kg)

### 地表数据 (surface)
- `pres.sfc` - 地表气压 (Surface pressure)
- `air.2m` - 2米气温 (2m air temperature)
- `uwnd.10m` - 10米纬向风 (10m zonal wind)
- `vwnd.10m` - 10米经向风 (10m meridional wind)

### 下载其他变量示例
```python
# 下载气温数据
files_air = download_ncep_data(
    variable='air',
    start_year=1979,
    end_year=2014,
    output_dir='./ncep_data/air'
)

# 下载位势高度
files_hgt = download_ncep_data(
    variable='hgt',
    start_year=1979,
    end_year=2014,
    output_dir='./ncep_data/hgt'
)
```